# B4_whats_new: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/bonus/B4_whats_new/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Estimate distances with one-bit RaBitQ and an explicit correction term.
Concepts: rotation, quantization and rescoring.
Run: uv run python modules/bonus/B4_whats_new/examples/01_from_scratch.py
Expected: assertions pass, then measured values or a clearly labeled illustrative result.
CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
Native profiles and optional packages are described in README.md.
"""

import numpy as np
from threadpoolctl import threadpool_limits

from tinyvec.rabitq import RaBitQ


def main() -> None:
    vectors = np.random.default_rng(42).normal(size=(500, 64)).astype(np.float32)
    with threadpool_limits(limits=1):
        index = RaBitQ(vectors)
        estimated = index.distances(vectors[0])
    assert abs(estimated[0]) < 1e-3
    print(
        {
            "first_candidates": np.argsort(estimated)[:5].tolist(),
            "code_and_rotation_bytes": index.payload_bytes,
            "self_distance": float(estimated[0]),
        }
    )
    print("Dense unpacking teaches the estimator; it is not a SIMD implementation.")


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/bonus/B4_whats_new/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Compare FAISS flat, rotated one-bit and four-bit RaBitQ.
    Concepts: rotation, quantization and rescoring.
    Run: uv run python modules/bonus/B4_whats_new/examples/02_production.py
    Expected: assertions pass, then measured values or a clearly labeled illustrative result.
    CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
    Native profiles and optional packages are described in README.md.
    """
    # course-extra: libs

    from common.research import rabitq_experiment, record

    def main() -> None:
        result = rabitq_experiment(native=True)
        print(result["native"])
        print(
            record(
                "rabitq-native",
                result,
                "python modules/bonus/B4_whats_new/examples/02_production.py",
            )
        )

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/bonus/B4_whats_new/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure seed variability and the accuracy cost before exact rescoring.
Concepts: rotation, quantization and rescoring.
Run: uv run python modules/bonus/B4_whats_new/examples/03_measure.py
Expected: assertions pass, then measured values or a clearly labeled illustrative result.
CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
Native profiles and optional packages are described in README.md.
"""

from common.research import rabitq_experiment, record


def main() -> None:
    result = rabitq_experiment()
    print(result["scratch"])
    print(
        record("rabitq-scratch", result, "python modules/bonus/B4_whats_new/examples/03_measure.py")
    )


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
import numpy as np

from tinyvec.rabitq import RaBitQ

x = np.random.default_rng(1).normal(size=(50, 16)).astype(np.float32)
q = RaBitQ(x)
print(q.distances(x[0])[0], q.payload_bytes)

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.